# 2. Can we trust this price chart?

A colleague sends a chart that appears to show an extraordinary winner. Your job is to decide whether the claim survives inspection.
You will produce **a comparison chart and a short data-quality note**.

This notebook is independent of module 1. The CSV contains **synthetic prices**, labeled AAPL, MSFT, and NVDA to connect to our portfolio example. These are classroom observations, not historical market quotes.

By the end, you should be able to sort before calculating changes, choose an informative plot, clean and parse text fields, and verify a repair with Python.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
%matplotlib inline
# Supplied setup: locate this repository even if Jupyter starts in a lecture folder.
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Reuse the three moves: select, calculate, summarize
**Retrieve from memory:** What does `table.loc[condition, ['column']]` do?
The new operation below, `read_csv`, turns a file into a DataFrame. `parse_dates` makes the date column usable as dates.

In [ ]:
data_path = course_root / 'Lectures/Lecture 2/data/classroom_prices.csv'
prices = pd.read_csv(data_path, parse_dates=['date'])
display(prices.head())
print('Rows:', len(prices), '| Assets:', prices['ticker'].unique())
prices.info()

### A three-price example you can check by hand
If a price moves 100 → 110 → 99, the returns are +10% and −10%.
The final cumulative return is **−1%**, not 0%. Returns compound.
`shift(1)` lines up the previous observation with today's row; `pct_change` is the shorter form.

In [ ]:
toy = pd.Series([100.0, 110.0, 99.0], name='price')
previous = toy.shift(1)
display(pd.DataFrame({'price': toy, 'previous': previous, 'return': toy / previous - 1}))
print('Same calculation:', toy.pct_change(fill_method=None).tolist())
print('Whole-period return:', toy.iloc[-1] / toy.iloc[0] - 1)

## 2. Choose a plot for a question
**Predict:** Does MSFT's higher share price mean a better performance?
Sort each asset by date before any change or rolling calculation. Calculate changes **within ticker**, never across an asset boundary.

In [ ]:
prices = prices.sort_values(['ticker', 'date']).reset_index(drop=True)
prices['return'] = prices.groupby('ticker')['close'].pct_change(fill_method=None)
print('The first return for each asset is undefined:')
display(prices.groupby('ticker').head(1)[['ticker', 'date', 'close', 'return']])

A **line chart** answers “how did it change?”
The first panel uses dollars. The second starts each asset at 100 so percentage changes are comparable.
`fig` is the whole figure; `axes[0]` and `axes[1]` are its two panels. We will reuse this pattern in the report next week.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
for ticker, rows in prices.groupby('ticker'):
    axes[0].plot(rows['date'], rows['close'], label=ticker)
    normalized = rows['close'] / rows['close'].iloc[0] * 100
    axes[1].plot(rows['date'], normalized, label=ticker)
axes[0].set(title='Share prices', xlabel='Date', ylabel='Price')
axes[1].set(title='Same starting point', xlabel='Date', ylabel='Indexed price (start = 100)')
for ax in axes:
    ax.legend()
plt.show()

### Worked example: the distribution of MSFT returns
A histogram counts observations in ranges of values. Multiply fractional returns by 100 to label the horizontal axis in percent.

In [ ]:
example_msft = prices.loc[prices['ticker'] == 'MSFT'].copy()
(example_msft['return'] * 100).plot.hist(bins=20, title='Synthetic MSFT daily returns', xlabel='Daily return (%)')
plt.show()

## 3. The impressive result contains a bad price
We deliberately damage a copy of the same observations. The supplied setup injects missing closes, a decimal error, ticker formatting, bad volume, and one repeated row.
**Predict:** Which issue could make a stock look like a spectacular winner if you stopped your analysis on the wrong day?

In [ ]:
raw = pd.read_csv(data_path, parse_dates=['date'])
raw.loc[2, 'ticker'] = ' aapl '
raw.loc[[4, 5, 120], 'close'] = np.nan
raw.loc[10, 'close'] *= 100
raw['volume'] = raw['volume'].astype(float)
raw.loc[12, 'volume'] = -100
raw.loc[14, 'volume'] = np.inf
raw = pd.concat([raw, raw.iloc[[3]]], ignore_index=True).sample(frac=1, random_state=7)
raw.plot.scatter(x='date', y='close', title='The vendor file: investigate before interpreting', figsize=(10, 3))
plt.show()

### Worked example: a close below the daily low
The close should lie between the day's low and high. In this small delivery, DEMO_A closes below its low. Select the inconsistent row with a mask.

In [ ]:
example_quotes = pd.DataFrame({
    'ticker': ['DEMO_A', 'DEMO_B'],
    'close': [98.0, 105.0], 'low': [99.0, 101.0], 'high': [103.0, 107.0],
})
example_bad_prices = example_quotes.loc[example_quotes['close'] < example_quotes['low'],
                                        ['ticker', 'close', 'low']]
display(example_bad_prices)

In [ ]:
print('Missing values:')
display(raw.isna().sum())
print('Ticker variants:', raw['ticker'].unique())
print('Non-finite volume:', (~np.isfinite(raw['volume'])).sum())
# Infinity is non-finite, but isna() does not count it as missing.

## 4. Can we use this vendor file to value our holdings?
You need to match a vendor's quotes to the tickers in a portfolio. But `aapl`, ` AAPL `, and `<b>AAPL</b>` will not match the same key yet, and `"price: 150.25"` cannot be multiplied by a share count.

The small synthetic export below has those problems. We will turn its labels and quotes into usable columns while keeping the original fields for comparison.

**Predict:** Which rows refer to the same holding? Which quotes would you want to investigate before using them?

In [ ]:
vendor_text = pd.DataFrame({
    'raw_symbol': [' nasdaq: aapl ', 'NASDAQ:<b>AAPL</b>', ' nasdaq : msft ', 'NYSE:JPM', None],
    'raw_name': [' Apple   Inc. ', 'APPLE INC.', 'Microsoft\tCorp.', ' JPMORGAN   CHASE ', None],
    'raw_quote': [' price: 150.25 ', 'PRICE :151.00', 'price: -5', 'price: unavailable', None],
})
display(vendor_text)
print('Distinct raw symbols:', vendor_text['raw_symbol'].nunique())

### Make the identifiers match
`.str` applies a text operation to each row. Remove the simple markup, trim the outside spaces with `strip`, and standardize case with `upper`. Then `split(':')` separates the exchange from the ticker.

Company names also have repeated spaces or tabs. Replacing `r'\s+'` with one space makes them readable. Use pandas' `string` dtype so a missing name stays missing.

In these examples, `regex=True` lets a replacement match a pattern, such as a run of spaces. Use `regex=False` when replacing literal text.

In [ ]:
example_text = vendor_text.copy()
symbol_text = (example_text['raw_symbol'].astype('string')
               .str.replace(r'<[^>]+>', '', regex=True)
               .str.strip().str.upper())
symbol_parts = symbol_text.str.split(':', n=1, expand=True)
example_text['exchange'] = symbol_parts[0].str.strip()
example_text['ticker'] = symbol_parts[1].str.strip()
example_text['company_name'] = (example_text['raw_name'].astype('string')
                                .str.strip().str.upper()
                                .str.replace(r'\s+', ' ', regex=True))
display(example_text[['raw_symbol', 'exchange', 'ticker', 'raw_name', 'company_name']])
print('Distinct cleaned tickers:', example_text['ticker'].nunique())

### Turn a quote into a number we can use
`str.extract` picks out the number after the `price:` label. The pattern below accepts extra spaces, a decimal point, and a sign. `pd.to_numeric` then converts the extracted text into a numeric column.

A failed conversion becomes missing with `errors='coerce'`. Keep it visible for review rather than replacing it with zero: a zero price would erase a holding's value.

Parsing and validation answer different questions. `price: -5` contains a number, but it is not a usable equity price. The pattern matches the whole field so an unrelated number in a vendor note is not mistaken for a quote.

In [ ]:
price_pattern = r'^price\s*:\s*([+-]?\d+(?:\.\d+)?)$'
quote_text = example_text['raw_quote'].astype('string').str.strip().str.lower()
example_price_text = quote_text.str.extract(price_pattern, expand=False)
example_text['price'] = pd.to_numeric(example_price_text, errors='coerce')
example_text['price_needs_review'] = example_text['price'].isna() | example_text['price'].le(0)
display(example_text[['raw_quote', 'price', 'price_needs_review']])
display(example_text.loc[example_text['price_needs_review']])
assert example_text.loc[0, 'price'] == 150.25
assert example_text.loc[2, 'price'] == -5
assert example_text.loc[[3, 4], 'price'].isna().all()
print('Quotes needing review:', example_text['price_needs_review'].sum())

### Clean labels do not resolve conflicting quotes
The two AAPL labels now match, but their quoted prices differ. Text cleanup helps us find that disagreement; it does not tell us which quote is authoritative.

Return to the price-history table. Normalize its tickers before looking for repeated `(ticker, date)` keys. An exact repeated record is different from two vendors disagreeing about a price.

In [ ]:
clean = raw.copy()
clean['ticker'] = clean['ticker'].str.strip().str.upper()
key = ['ticker', 'date']
display(clean.loc[clean.duplicated(key, keep=False)].sort_values(key))
clean = clean.drop_duplicates().sort_values(key).reset_index(drop=True)
assert not clean.duplicated(key).any(), 'Conflicting records need a source-selection rule.'

## 5. Make the repair explicit
First flag values we cannot use; then decide what information justifies a replacement.
In this controlled example **only close was corrupted**, so open/high/low are trustworthy.
With real data, a bounds violation alone does not tell us which field is wrong.

A large but internally consistent return is a **review flag**, not automatic permission to erase an observation.

In [ ]:
bad_close = (~np.isfinite(clean['close']) | (clean['close'] <= 0)
             | (clean['close'] < clean['low']) | (clean['close'] > clean['high']))
clean['close_repaired'] = bad_close
clean.loc[bad_close, 'close'] = np.nan
print('Close values to investigate:', bad_close.sum())

### Why “just forward fill” can still be wrong
On these two days, yesterday's close is outside today's range. Forward fill removes the null but creates an impossible OHLC row.
**Discuss:** what do we know now, and what would we have known at the time?

In [ ]:
gap_example = pd.DataFrame({'low': [99.0, 109.0], 'high': [101.0, 111.0], 'close': [100.0, np.nan]})
gap_example['ffill_candidate'] = gap_example['close'].ffill()
gap_example['candidate_in_range'] = gap_example['ffill_candidate'].between(gap_example['low'], gap_example['high'])
gap_example

For this example, use a prior close only if it lies inside today's range; otherwise use today's range midpoint.
Keep a repair flag because an estimate is not a recovered trade. Do not use backward fill or interpolation in a feature that must be known before later data arrives.
The next cell is a **worked repair**. Read it in three steps: candidate → condition → replacement.

In [ ]:
previous = clean.groupby('ticker')['close'].ffill()
within_range = previous.between(clean['low'], clean['high'])
midpoint = (clean['high'] + clean['low']) / 2
replacement = previous.where(within_range, midpoint)
clean['close'] = clean['close'].fillna(replacement)
invalid_volume = ~np.isfinite(clean['volume']) | (clean['volume'] < 0)
clean['volume_repaired'] = invalid_volume
clean.loc[invalid_volume, 'volume'] = 0  # A flagged placeholder for unknown volume, not proof of no trading.
display(clean.loc[clean['close_repaired'] | clean['volume_repaired']])

### Check the result with executable statements
An `assert` expresses something that must be true. It stops execution when that statement is false.
**Predict:** Which check would catch the forward-fill mistake above?
A return uses two closes, so a repaired close can affect two consecutive returns.

In [ ]:
price_cols = ['open', 'high', 'low', 'close']
assert np.isfinite(clean[price_cols]).all().all()
assert clean[price_cols].gt(0).all().all()
assert clean['close'].between(clean['low'], clean['high']).all()
assert clean['open'].between(clean['low'], clean['high']).all()
assert np.isfinite(clean['volume']).all() and clean['volume'].ge(0).all()
assert not clean.duplicated(key).any()
clean['return'] = clean.groupby('ticker')['close'].pct_change(fill_method=None)
clean['return_affected'] = clean['close_repaired'] | clean.groupby('ticker')['close_repaired'].shift(1, fill_value=False)
print('Checks passed. We still need to explain the estimates.')

## 6. Document a missing-volume repair
This small delivery has one missing volume. A flag records where we use a zero placeholder; the flag distinguishes unknown volume from a genuine zero.

In [ ]:
example_volumes = pd.DataFrame({'ticker': ['DEMO_A', 'DEMO_B', 'DEMO_C'],
                               'volume': [100.0, np.nan, 200.0]})
example_volumes['volume_repaired'] = example_volumes['volume'].isna()
example_volumes.loc[example_volumes['volume_repaired'], 'volume'] = 0
display(example_volumes)
print('DEMO_B volume is unknown; its zero is a flagged placeholder.')

### Save an inspectable result
We keep raw data in memory and save the cleaned table **with flags**.
The line chart below describes synthetic data; the quality note should describe the repair assumptions before anyone interprets it.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
for ticker, rows in clean.groupby('ticker'):
    ax.plot(rows['date'], rows['close'] / rows['close'].iloc[0] * 100, label=ticker)
ax.set(title='After documented repairs', xlabel='Date', ylabel='Indexed price (start = 100)')
ax.legend()
output_dir = course_root / 'Lectures/Lecture 2/outputs'
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir / 'clean_prices.png', dpi=150, bbox_inches='tight')
clean.to_csv(output_dir / 'clean_prices_demo.csv', index=False)
plt.show()

## Exit check
Explain: (1) why the first return is missing, (2) why sorting/grouping comes before returns, and (3) why no nulls does not guarantee valid data.
Next session turns these habits into **SQL retrieval and a reusable report**. Later modeling classes reuse `shift`, rolling calculations, and the distinction between past and future information.

Optional: [plotting styles, scatter/box/KDE and correlations](reference/4_visualization_seeing_your_data.ipynb);
[Z-score/IQR, drop/fill/interpolate, string methods](reference/5_data_cleaning_fundamentals.ipynb).
[Homework 2](hw2.md) remains due October 29.

## Exercises
Use the tables and functions above to complete these tasks. Replace the placeholders with your code, then run the checks.

### Your turn C: one asset, one distribution
Select NVDA's rows into `your_nvda`. Plot its daily returns as a histogram, with **percent** on the horizontal axis.
A histogram asks “how often did moves of this size occur?” rather than “when did they occur?”
**Hint:** `(your_nvda['return'] * 100).plot.hist(bins=20)`.
**Stretch:** compare AAPL on another axis with the same horizontal scale. Avoid claiming that a synthetic sample predicts future risk.

In [ ]:
your_nvda = None
# Select NVDA, create a histogram, and label the units.

In [ ]:
if your_nvda is None:
    print('Try C: select by ticker, then plot return * 100.')
else:
    assert len(your_nvda) == 120 and your_nvda['ticker'].eq('NVDA').all()
    assert your_nvda['return'].isna().sum() == 1
    print('Selection passed. Check that the histogram title identifies the asset and the axis states percent.')

### Your turn D: give the colleague evidence
Write a mask for `close > high`, then select the date, ticker, close, and high of the flagged rows into `your_bad_prices`.
**Hint:** `raw.loc[raw['close'] > raw['high'], [...]]`.
A daily **high** is the largest traded price that day, so a close above it is inconsistent.
**Stretch:** why does `close > high` fail to find a missing close?

In [ ]:
your_bad_prices = None

In [ ]:
if your_bad_prices is None:
    print('Try D. The injected decimal error should produce exactly one row.')
else:
    assert len(your_bad_prices) == 1
    assert your_bad_prices['close'].gt(your_bad_prices['high']).all()
    print('Fault located. Explain why this is stronger evidence than just a large return.')

### Your turn: repair a new delivery with negative volume
Make a copy called `new_delivery`, set its first row's volume to −5, and then:
1. Identify invalid volumes with a mask (include both negative and non-finite values).
2. Mark them and replace them with the documented zero placeholder.
3. Check the result and write one sentence describing what is still unknown.

Assign the repaired table to `your_delivery` and your sentence to `your_quality_note`.
**Hint:** reuse a pattern from above; do not change `raw` or `clean`.
**Stretch:** what else would change if this were the first **price** observation of a ticker, rather than volume?

In [ ]:
new_delivery = clean.copy()
new_delivery.loc[new_delivery.index[0], 'volume'] = -5
your_delivery = None
your_quality_note = ''

In [ ]:
if your_delivery is None:
    print('Complete the new-delivery repair and write a quality note.')
else:
    assert len(your_delivery) == len(clean)
    assert np.isfinite(your_delivery['volume']).all() and your_delivery['volume'].ge(0).all()
    assert your_delivery.loc[your_delivery.index[0], 'volume'] == 0
    assert your_delivery.loc[your_delivery.index[0], 'volume_repaired']
    assert your_quality_note.strip()
    print('Repair checks passed. Explain why the repaired zero is not an observed zero.')

### Your turn: clean a second vendor delivery
A second vendor sends JPM and NVDA quotes for a portfolio valuation. Prepare the fields for matching and calculation. Keep every row and the three raw columns, then add:

1. `exchange` and `ticker`: remove the simple markup, split on `:`, trim whitespace, and uppercase the labels.
2. `company_name`: uppercase and trim the names, then collapse repeated whitespace to one space.
3. `price`: extract the number from a field that follows the `price:` format and convert it to a numeric dtype. Keep missing or malformed fields missing.
4. `price_needs_review`: flag missing, malformed, or nonpositive prices.

Assign the result to `your_text_clean`. In `your_text_note`, explain which quotes you would hold back from the valuation and why. Preserve missing tickers as missing values.

**Hint:** use `.astype('string')`, the `.str` methods above, and `pd.to_numeric`. You can reuse `price_pattern`.
**Stretch:** explain why grabbing the first number from `batch 7 price:120` would be wrong. Add `price: -12.50` as another row and check that the sign survives parsing.

In [ ]:
practice_text = pd.DataFrame({
    'raw_symbol': [' nyse: jpm ', 'NYSE:<i>JPM</i>', ' nasdaq : nvda ', 'NASDAQ:NVDA', None],
    'raw_name': [' JPMorgan   Chase ', 'jpmorgan chase', ' NVIDIA\tCorp. ', 'nvidia  corp.', None],
    'raw_quote': [' PRICE : 210.50 ', 'price:211', 'price:0', 'batch 7 price:120', None],
})
display(practice_text)
your_text_clean = None
your_text_note = ''

In [ ]:
if your_text_clean is None:
    print('Clean the text columns, parse the quotes, and write a note about a flagged row.')
else:
    assert isinstance(your_text_clean, pd.DataFrame)
    assert len(your_text_clean) == len(practice_text), 'Keep all rows so rejected values remain inspectable.'
    pd.testing.assert_frame_equal(your_text_clean[practice_text.columns], practice_text)
    assert your_text_clean['exchange'].iloc[:4].tolist() == ['NYSE', 'NYSE', 'NASDAQ', 'NASDAQ']
    assert your_text_clean['ticker'].iloc[:4].tolist() == ['JPM', 'JPM', 'NVDA', 'NVDA']
    assert your_text_clean['company_name'].iloc[:4].tolist() == ['JPMORGAN CHASE', 'JPMORGAN CHASE', 'NVIDIA CORP.', 'NVIDIA CORP.']
    assert your_text_clean.loc[4, ['exchange', 'ticker', 'company_name']].isna().all(), 'Keep missing text missing.'
    assert pd.api.types.is_numeric_dtype(your_text_clean['price']), 'The parsed prices must be numeric.'
    np.testing.assert_allclose(your_text_clean['price'].iloc[:3].to_numpy(dtype=float), [210.5, 211, 0])
    assert your_text_clean['price'].iloc[3:].isna().all(), 'An unrelated number is not a valid quote.'
    assert your_text_clean['price_needs_review'].tolist() == [False, False, True, True, True]
    assert your_text_note.strip(), 'Explain what a flagged value means.'
    print('Text-cleaning checks passed. Explain why a numeric zero and a failed parse both need review.')